腳本簡易操作手冊 (SOP)
本手冊將引導您如何快速設定並執行此 Python 分析腳本。

1. 腳本目標
此腳本用於計算分子動力學模擬中，每一顆鉀離子 (POT) 與所有鳥嘌呤 (GUA) 質心之間的距離，並利用多核心處理器加速計算。

2. 環境準備
確保您的 Python 環境已安裝必要的函式庫。打開您的終端機（Terminal 或命令提示字元），執行以下指令：

pip install MDAnalysis numpy

3. 修改腳本設定
在執行前，您需要修改腳本中的檔案路徑。請用文字編輯器打開您的 .py 檔案。

找到以下區塊：

# 設定文件路徑
pdb_file = "/home/asher/G4/G4_ion_center.pdb"
dcd_files = [
    "/work/asher/DCD_place/ion_center/G4_ion.dcd",
    "/work/asher/DCD_place/ion_center/G4_ion1000.dcd",
]

進行修改：

將 pdb_file 的值（"/home/asher/..."）替換成您自己 PDB 檔案的 絕對路徑或相對路徑。

將 dcd_files 列表中的路徑，替換成您所有 DCD 軌跡檔的路徑。

4. 執行腳本
開啟終端機。

使用 cd 指令切換到您存放 Python 腳本的資料夾。

執行以下指令（請將 your_script_name.py 換成您實際的檔名）：

python your_script_name.py

5. 查看結果
執行完畢後，終端機將顯示：

距離計算完成，結果保存至 distances_1.txt

(註：檔名中的數字 1 可能會因已存在檔案而遞增)

輸出檔案格式：
產生的 distances_X.txt 檔案內容如下，共三欄，以空格分隔：

[影格編號] [離子索引] [距離]

影格編號: 來自 DCD 軌跡檔的影格（Frame）編號。

離子索引: 鉀離子的編號（從 0 開始）。

距離: 該離子在該影格與鳥嘌呤質心的距離（單位為 Ångström）。

In [ ]:
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
from MDAnalysis.transformations import unwrap

# 定義要載入的 PDB 和 DCD 文件路徑
pdb_file = "/home/asher/G4/G4_ion_center.psf"
dcd_file = "/work/asher/DCD_place/ion_center/G4_ion.dcd"

# 載入 Universe，這包含了系統的拓撲 (PDB) 和軌跡 (DCD)
u = mda.Universe(pdb_file, dcd_file)

# 選擇 DNA 和離子的原子
# 這裡的選擇根據 resname，具體可以根據實際情況調整
dna = u.select_atoms("resname GUA")  # DNA 的殘基名稱
ion = u.select_atoms("resname POT")  # 假設是鉀離子 (POT)

u.trajectory.add_transformations(unwrap(u.atoms))

# 用來儲存每一幀的最小距離
distances = []

# 遍歷每一幀，計算 DNA 和離子之間的距離
for ts in u.trajectory:
    # 計算每一幀中 DNA 和離子之間的最小距離
    min_distance = np.min(mda.lib.distances.distance_array(dna.positions, ion.positions))
    distances.append(min_distance)

# 繪製距離隨時間的變化圖
plt.figure(figsize=(10, 6))
plt.plot(distances, label="Ion-DNA Minimum Distance")
plt.xlabel("Frame")
plt.ylabel("Distance (Å)")
plt.title("Ion-DNA Distance Over Time")
plt.legend()
plt.show()


In [ ]:
from multiprocessing import Pool
import MDAnalysis as mda
import numpy as np
import os

# 設定文件路徑
pdb_file = "/home/asher/G4/G4_ion_center.pdb"
dcd_files = [
    "/work/asher/DCD_place/ion_center/G4_ion.dcd",
    "/work/asher/DCD_place/ion_center/G4_ion1000.dcd",
]

# 檢查文件是否存在，並自動生成遞增的文件名
file_index = 1
filename = f"distances_{file_index}.txt"

while os.path.exists(filename):
    file_index += 1
    filename = f"distances_{file_index}.txt"

# 計算單顆離子距離的函數
def calculate_distances(ion_idx):
    u = mda.Universe(pdb_file, dcd_files, guess_bonds=True)  # 啟用猜測鍵結
    pfos_sel = u.select_atoms("resname GUA")
    k_ion = u.select_atoms("resname POT")[ion_idx]
    distances = []

    # 遍歷軌跡的每一幀
    for ts in u.trajectory[::50]:  # 每 50 幀取一次
        # 確保坐標解包裹
        u.atoms.unwrap()

        # 計算分子的質心
        pfos_com = pfos_sel.center_of_mass()

        # 計算 K+ 離子的坐標
        k_pos = k_ion.position

        # 計算它與質心的距離
        distance = np.linalg.norm(k_pos - pfos_com)

        # 存儲距離
        distances.append((ts.frame, ion_idx, distance))

    return distances

# 主程序部分
if __name__ == "__main__":
    # 加載 Universe 並獲取 K+ 離子的總數
    u = mda.Universe(pdb_file, dcd_files, guess_bonds=True)  # 啟用猜測鍵結
    num_ions = len(u.select_atoms("resname POT"))

    # 創建一個進程池來進行平行計算
    with Pool() as pool:
        results = pool.map(calculate_distances, range(num_ions))

    # 將計算結果寫入文件
    with open(filename, "w") as outfile:
        for distances in results:
            for frame, ion_idx, distance in distances:
                outfile.write(f"{frame} {ion_idx} {distance:.3f}\n")

    print(f"距離計算完成，結果保存至 {filename}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# 設定文件路徑
filename = "/home/asher/jubook/distances_1.txt"  # 根據實際情況更改

# 檢查文件是否存在
if not os.path.exists(filename):
    print(f"Error: File '{filename}' not found.")
else:
    # 讀取距離數據
    data = np.loadtxt(filename)

    # 幀數、離子編號、距離數據
    frames = data[:, 0]
    ion_indices = data[:, 1]
    distances = data[:, 2]

    # 找出所有的離子編號
    unique_ions = np.unique(ion_indices)

    # 為每個離子畫出它的距離隨時間變化的圖
    plt.figure(figsize=(10, 6))
    for ion_idx in unique_ions:
        mask = ion_indices == ion_idx
        plt.plot(frames[mask], distances[mask], label=f'Ion {int(ion_idx)}')

    # 圖表設置
    plt.xlabel("Frame", fontsize=14)
    plt.ylabel("Distance to Center of Mass (Å)", fontsize=14)
    plt.title("Distance of K+ Ions to DNA Center of Mass Over Time", fontsize=16)
    plt.legend(loc='best', fontsize=10)
    plt.grid(True)

    # 保存圖表
    plt.savefig("/home/asher/distance_new")
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
filename = "/home/asher/G4/tcl/distances_11.txt"

data = np.loadtxt(filename)


frames = data[:, 0]
k_ions_indices = data[:, 1]
distances = data[:, 2]


plt.figure(figsize=(10, 6))


for ion_index in np.unique(k_ions_indices):
    ion_distances = distances[k_ions_indices == ion_index]
    ion_frames = frames[k_ions_indices == ion_index]
    plt.plot(ion_frames, ion_distances, label=f'K+ Ion {int(ion_index)}')
    plt.xlabel('Frame')
    plt.ylabel('Distance (\u00c5)')
    plt.title('Distance Between Ion and DNA Time')
    
    plt.legend()
    
    plt.show()